In [1]:
import pandas as pd
import re

def modify_inp_file(inp_file_path, pump_settings_file, output_inp_file_path):
    # Read pump settings CSV
    df_pump_settings = pd.read_csv(pump_settings_file, index_col="Name")
    pumps_off = df_pump_settings[df_pump_settings["rep_0"] == 0].index.tolist()

    # Read the .inp file
    with open(inp_file_path, "r") as file:
        lines = file.readlines()

    # Flags to detect the [PUMPS] section
    in_pumps_section = False
    modified_lines = []

    # Loop through each line
    for line in lines:
        # Detect section start
        if line.strip() == "[PUMPS]":
            in_pumps_section = True
            modified_lines.append(line)
            continue
        # Detect section end
        elif line.strip().startswith("[") and in_pumps_section:
            in_pumps_section = False

        # Modify pump lines if pump is in the 'pumps_off' list
        if in_pumps_section and not line.strip().startswith(";") and len(line.strip()) > 0:
            parts = line.split()
            if len(parts) >= 7:  # Ensure line has at least 7 columns
                pump_name = parts[0]
                if pump_name in pumps_off:
                    parts[5] = "1000"  # Set Startup to 1000 ft
                    line = " ".join(parts) + "\n"

        modified_lines.append(line)

    # Save the modified .inp file
    with open(output_inp_file_path, "w") as file:
        file.writelines(modified_lines)

    print(f"Modified .inp file saved as '{output_inp_file_path}'")


# File paths
inp_file_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event.inp"
pump_settings_file = "pump_outages_single_iteration.csv"
output_inp_file_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_modified_pumps.inp"

# Run the function
modify_inp_file(inp_file_path, pump_settings_file, output_inp_file_path)


Modified .inp file saved as 'C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_modified_pumps.inp'


In [3]:
import os
import traceback
import pandas as pd
from datetime import datetime
from pyswmm import Simulation, Nodes, Links

# File paths
input_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_modified_new.inp"
hotstart_hsf_file = "hotstart_harris_no_event_modified.hsf"
output_csv_file = "node_depths_daily_no_event_modified.csv"

# Run SWMM simulation
def run_swmm_simulation(input_path, start_time, end_time, hotstart_hsf_file, output_csv_file):
    print(f"Running SWMM simulation from {start_time} to {end_time}...")

    try:
        restart_simulation = os.path.exists(hotstart_hsf_file)

        with Simulation(input_path) as sim:
            if restart_simulation:
                print("Hotstart file found. Applying initial conditions...")
                sim.use_hotstart(hotstart_hsf_file)
            else:
                print("No hotstart file found. Starting simulation from scratch...")

            sim.start_time = start_time
            sim.end_time = end_time
            sim.step_advance(900)  # 15-minute steps
                   
            # Store node references for fast access
            nodes = {node.nodeid: node for node in Nodes(sim)}

            # Manually advance one step to apply hotstart
            print("\nAdvancing one step to apply hotstart...")
            next(sim)

            # Dictionary to store daily depth variations
            daily_depths = {}

            print("Starting full simulation...")
            for step in sim:
                current_time = sim.current_time
                date_str = str(current_time.date())

                # Log progress every 6 hours
                if current_time.hour % 6 == 0 and current_time.minute == 0:
                    print(f"Simulation Time: {current_time}")

                # Store depth variations throughout the day
                for node_id, node in nodes.items():
                    if (date_str, node_id) not in daily_depths:
                        daily_depths[(date_str, node_id)] = []
                    daily_depths[(date_str, node_id)].append(node.depth)  # Collect depth changes

            # Process depths: Calculate min, max, and average per day
            node_depths = []
            for (date, node_id), depth_list in daily_depths.items():
                node_depths.append([date, node_id, min(depth_list), max(depth_list), sum(depth_list) / len(depth_list)])

            # Debug print before saving
            print("\nChecking recorded node depths before saving to CSV:")
            if node_depths:
                for entry in node_depths[:5]:  # Print first 5 entries
                    print(entry)
            else:
                print("No depths recorded! Check simulation step timing.")

            # Convert recorded depths to DataFrame & Save as CSV
            df_depths = pd.DataFrame(node_depths, columns=["Date", "Node ID", "Min Depth", "Max Depth", "Avg Depth"])
            df_depths.to_csv(output_csv_file, index=False)
            print(f"Daily node depths saved to {output_csv_file}")

    except Exception as e:
        print("Error during simulation:", e)
        traceback.print_exc()

# Main workflow
if __name__ == "__main__":
    print("Starting optimized SWMM workflow...")

    # Define simulation period
    full_start_time = datetime(2021, 6, 15, 0, 1, 0)  # Start date
    full_end_time = datetime(2021, 6, 23, 0, 0, 0)    # End date

    # Run simulation on modified .inp file
    run_swmm_simulation(
        input_path,
        full_start_time,
        full_end_time,
        hotstart_hsf_file,
        output_csv_file,
    )


Starting optimized SWMM workflow...
Running SWMM simulation from 2021-06-15 00:01:00 to 2021-06-23 00:00:00...
Hotstart file found. Applying initial conditions...

Advancing one step to apply hotstart...
Starting full simulation...

Checking recorded node depths before saving to CSV:
['2021-06-15', '100100', 0.011291898014146658, 0.03402542551640346, 0.01787332976400981]
['2021-06-15', '100200', 0.0319895470199496, 0.05711999444437871, 0.042488117328505576]
['2021-06-15', '100300', 0.019581668697804373, 0.04267671233941932, 0.02937178810377842]
['2021-06-15', '11521', 1.8347850043651466, 2.9760010525409766, 2.2108597969802233]
['2021-06-15', '11523', 1.8357635681063242, 4.359127720383055, 2.660138024329745]
Daily node depths saved to node_depths_daily_no_event_modified.csv


In [1]:
import pandas as pd
import re

def modify_inp_file(inp_file_path, pump_settings_file, output_inp_file_path):
    # Read pump settings CSV
    df_pump_settings = pd.read_csv(pump_settings_file, index_col="Name")
    pumps_off = df_pump_settings[df_pump_settings["rep_0"] == 0].index.tolist()

    # Read the .inp file
    with open(inp_file_path, "r") as file:
        lines = file.readlines()

    # Flags to detect the [PUMPS] section
    in_pumps_section = False
    modified_lines = []

    # Loop through each line
    for line in lines:
        # Detect section start
        if line.strip() == "[PUMPS]":
            in_pumps_section = True
            modified_lines.append(line)
            continue
        # Detect section end
        elif line.strip().startswith("[") and in_pumps_section:
            in_pumps_section = False

        # Modify pump lines if pump is in the 'pumps_off' list
        if in_pumps_section and not line.strip().startswith(";") and len(line.strip()) > 0:
            parts = line.split()
            if len(parts) >= 7:  # Ensure line has at least 7 columns
                pump_name = parts[0]
                if pump_name in pumps_off:
                    parts[5] = "1000"  # Set Startup to 1000 ft
                    line = " ".join(parts) + "\n"

        modified_lines.append(line)

    # Save the modified .inp file
    with open(output_inp_file_path, "w") as file:
        file.writelines(modified_lines)

    print(f"Modified .inp file saved as '{output_inp_file_path}'")


# File paths
inp_file_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event.inp"
pump_settings_file = "pump_outages_single_iteration_Pumps_Off.csv"
output_inp_file_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_all_depths_1000.inp"

# Run the function
modify_inp_file(inp_file_path, pump_settings_file, output_inp_file_path)


Modified .inp file saved as 'C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_all_depths_1000.inp'


In [1]:
import os
import traceback
import pandas as pd
from datetime import datetime
from pyswmm import Simulation, Nodes, Links

# File paths
input_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_alldepths_1000.inp"
hotstart_hsf_file = "hotstart_harris_no_event_pumps_off.hsf"
output_csv_file = "node_depths_daily_no_event_pumps_off.csv"

# Run SWMM simulation
def run_swmm_simulation(input_path, start_time, end_time, hotstart_hsf_file, output_csv_file):
    print(f"Running SWMM simulation from {start_time} to {end_time}...")

    try:
        restart_simulation = os.path.exists(hotstart_hsf_file)

        with Simulation(input_path) as sim:
            if restart_simulation:
                print("Hotstart file found. Applying initial conditions...")
                sim.use_hotstart(hotstart_hsf_file)
            else:
                print("No hotstart file found. Starting simulation from scratch...")

            sim.start_time = start_time
            sim.end_time = end_time
            sim.step_advance(900)  # 15-minute steps
                   
            # Store node references for fast access
            nodes = {node.nodeid: node for node in Nodes(sim)}

            # Manually advance one step to apply hotstart
            print("\nAdvancing one step to apply hotstart...")
            next(sim)

            # Dictionary to store daily depth variations
            daily_depths = {}

            print("Starting full simulation...")
            for step in sim:
                current_time = sim.current_time
                date_str = str(current_time.date())

                # Log progress every 6 hours
                if current_time.hour % 6 == 0 and current_time.minute == 0:
                    print(f"Simulation Time: {current_time}")

                # Store depth variations throughout the day
                for node_id, node in nodes.items():
                    if (date_str, node_id) not in daily_depths:
                        daily_depths[(date_str, node_id)] = []
                    daily_depths[(date_str, node_id)].append(node.depth)  # Collect depth changes

            # Process depths: Calculate min, max, and average per day
            node_depths = []
            for (date, node_id), depth_list in daily_depths.items():
                node_depths.append([date, node_id, min(depth_list), max(depth_list), sum(depth_list) / len(depth_list)])

            # Debug print before saving
            print("\nChecking recorded node depths before saving to CSV:")
            if node_depths:
                for entry in node_depths[:5]:  # Print first 5 entries
                    print(entry)
            else:
                print("No depths recorded! Check simulation step timing.")

            # Convert recorded depths to DataFrame & Save as CSV
            df_depths = pd.DataFrame(node_depths, columns=["Date", "Node ID", "Min Depth", "Max Depth", "Avg Depth"])
            df_depths.to_csv(output_csv_file, index=False)
            print(f"Daily node depths saved to {output_csv_file}")

    except Exception as e:
        print("Error during simulation:", e)
        traceback.print_exc()

# Main workflow
if __name__ == "__main__":
    print("Starting optimized SWMM workflow...")

    # Define simulation period
    full_start_time = datetime(2021, 6, 15, 0, 1, 0)  # Start date
    full_end_time = datetime(2021, 6, 23, 0, 0, 0)    # End date

    # Run simulation on modified .inp file
    run_swmm_simulation(
        input_path,
        full_start_time,
        full_end_time,
        hotstart_hsf_file,
        output_csv_file,
    )


Starting optimized SWMM workflow...
Running SWMM simulation from 2021-06-15 00:01:00 to 2021-06-23 00:00:00...
Hotstart file found. Applying initial conditions...

Advancing one step to apply hotstart...
Starting full simulation...

Checking recorded node depths before saving to CSV:
['2021-06-15', '100100', 0.011294256571525066, 0.03402068827199697, 0.01783699342089341]
['2021-06-15', '100200', 0.0319861386568185, 0.057111143629982555, 0.042482679560835034]
['2021-06-15', '100300', 0.019524535797501325, 0.042669514432916944, 0.02936735307269004]
['2021-06-15', '11521', 15.825538027748522, 16.67156953403756, 16.14830606193313]
['2021-06-15', '11523', 17.526955963975915, 18.462933403473066, 17.831112366676184]
Daily node depths saved to node_depths_daily_no_event_pumps_off.csv
